# Colab validator (llama3.1:8b)

Runs the **validator** stages on Colab's free GPU, inside this notebook (no tunnel). Discovery and extraction stay local on qwen.
Step-by-step clicks and the files to upload or download: `docs/colab_steps.md` in the repo.

Run the cells in order. After a disconnect, run them again from the top: done steps are skipped and every LLM call is cached on Drive, so nothing is lost.

In [ ]:
# Step 0: settings. Change STAGE only.
# night2: benchmarks, judge + agent | night3: benchmarks, self-consistency | real: ds101/ml101
STAGE = "night2"
COMMIT = "1052047bc74fbb717e5d2ea2384dae8234d47089"  # pinned repo commit
MODEL = "llama3.1:8b"  # validator model (discovery/extraction ran locally on qwen2.5:7b-instruct)
DRIVE_DIR = "/content/drive/MyDrive/cdlp"
REPO_URL = "https://github.com/shreeya-g10/concept-dependency-learning-path.git"
REPO = "/content/cdlp"
assert STAGE in ("night2", "night3", "real"), STAGE
assert len(COMMIT) == 40, "COMMIT must be a full 40-character commit SHA"

In [ ]:
# Step 1: is a GPU present? (Runtime > Change runtime type > T4 GPU)
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU > Save, then rerun.")
GPU = gpu.stdout.strip()
print("GPU:", GPU)

In [ ]:
# Step 2: install Ollama, start it, pull the model (rerun this cell after a runtime restart)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import time

import requests

server = subprocess.Popen(
    ["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT
)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Ollama did not start: see /content/ollama.log")
!ollama pull {MODEL}
!ollama --version

In [ ]:
# Step 3: clone the repo at the pinned commit; install only what the validator needs
import os

if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
!git -C {REPO} fetch -q origin
!git -C {REPO} checkout -q {COMMIT}
!git -C {REPO} log --oneline -1
!pip install -q requests python-dotenv jsonschema networkx numpy
if STAGE == "real":
    !pip install -q sentence-transformers  # the evidence check retrieves course chunks

In [ ]:
# Step 4: Drive keeps outputs/ and the LLM cache, so a disconnect loses nothing
import shutil
import zipfile

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/cache/llm", exist_ok=True)
zip_path = f"{DRIVE_DIR}/outputs.zip"
if os.path.exists(zip_path):
    # add newly uploaded local outputs; never overwrite a file already on Drive
    with zipfile.ZipFile(zip_path) as z:
        new = [m for m in z.namelist() if not os.path.exists(f"{DRIVE_DIR}/{m}")]
        z.extractall(DRIVE_DIR, members=new)
    os.replace(zip_path, f"{zip_path}.added")
    print(f"added {len(new)} files from outputs.zip")
if not os.path.isdir(f"{DRIVE_DIR}/outputs"):
    raise SystemExit(f"Upload outputs.zip to {DRIVE_DIR} first (docs/colab_steps.md).")
for link, target in (
    (f"{REPO}/outputs", f"{DRIVE_DIR}/outputs"),
    (f"{REPO}/.cache/llm", f"{DRIVE_DIR}/cache/llm"),
):
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.exists(link):
        shutil.rmtree(link)
    os.symlink(target, link)
os.environ.update(
    LLM_PROVIDER="ollama",
    LLM_BASE_URL="http://localhost:11434",
    LLM_VALIDATOR_MODEL=MODEL,
    LLM_MODEL=MODEL,
)
print("outputs on Drive:", sorted(os.listdir(f"{DRIVE_DIR}/outputs")))
print("cached LLM calls on Drive:", len(os.listdir(f"{DRIVE_DIR}/cache/llm")))

In [ ]:
# Step 5: seconds per call (10 short calls, throwaway cache)
%cd {REPO}
!python scripts/colab_run.py speed --model {MODEL} --calls 10

In [ ]:
# Step 6: run the stage; GPU, Ollama version and model digest go to outputs/colab_environment.json.
# After a disconnect: run steps 0-4, then this cell again. Done steps are skipped; calls are cached.
%cd {REPO}
!python scripts/colab_run.py run --stage {STAGE} --model {MODEL} --out-dir {DRIVE_DIR}/outputs

In [ ]:
# Step 7: progress (any time)
!python {REPO}/scripts/colab_run.py progress --stage {STAGE}

In [ ]:
# Step 8: zip the results (and the cache) on Drive for download
import datetime

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
results = shutil.make_archive(
    f"{DRIVE_DIR}/results_{STAGE}_{stamp}", "zip", root_dir=DRIVE_DIR, base_dir="outputs"
)
cache = shutil.make_archive(
    f"{DRIVE_DIR}/cache_{STAGE}_{stamp}", "zip", root_dir=f"{DRIVE_DIR}/cache", base_dir="llm"
)
print("Download from Drive:", results, cache, sep="\n  ")